# ЛР №4 — Теория к лабораторной работе
## «Распределённая работа с базой данных и транзакциями»

**Предметная область (вариант 14):** «Заказ и доставка цветов» — продолжение ЛР №1…ЛР №3.
**Общий ресурс:** `Bouquet.AvailableCount` — количество доступных экземпляров букета.
**Составная бизнес-операция:** создание заказа + уменьшение остатка букета (строка 14 вариантов).
**Проект:** `Лабы/Лаб4` (расширение решения `FlowerShop.sln` из ЛР №3).

> ЛР №4 сдаётся **отдельно** от ЛР №3 и продолжает тот же проект.


### Содержание

| № | Раздел | Что доказывает / что даёт |
|---|---|---|
| 1 | Архитектура Client → Server → Database | почему клиент не ходит в БД напрямую |
| 2 | БД, СУБД и выбор СУБД | где физически хранятся данные предметной области |
| 3 | Проектирование базы данных | таблицы, ключи, ограничения целостности |
| 4 | Подключение сервера к БД | операции чтения и изменения через сервер |
| 5 | Два уровня согласованности | lock (код сервера) и СУБД (данные) |
| 6 | Критическая секция и `lock` | почему `SUCCESS`+`SUCCESS` при `Resource = 1` — ошибка |
| 7 | Транзакции, COMMIT, ROLLBACK | атомарность составной операции |
| 8 | `lock` и транзакция: разные задачи | почему один механизм не заменяет другой |
| 9 | Задание: 4 части и требования | что именно реализовать |
| 10 | Таблицы результатов | что заполнять для защиты |
| 11 | Порядок экспериментов | пошаговая процедура для защиты |
| 12 | Карта «требование → код → результат» | быстрая навигация по проекту |
| 13 | Ответы на контрольные вопросы (1–16) | подготовка к устной защите |
| 14 | Ключевые выводы | итог работы |


---
## 1. Архитектура Client → Server → Database

В ЛР №1–№3 основное состояние приложения хранилось **в памяти сервера**
(`InMemoryDataStore`). В ЛР №4 состояние переносится в базу данных, при этом
схема общения остаётся трёхзвенной:

```
ЛР №1–№3:   Client ──────────────► Server (память: InMemoryDataStore)

ЛР №4:      Client ──запрос──────► Server ──SQL──► Database
            Client ◄──ответ────── Server ◄──данные─ Database
```

**Клиент не обращается к базе данных напрямую.** Клиент отправляет запрос серверу, а сервер:

1. принимает запрос;
2. выполняет необходимые действия (бизнес-логика);
3. обращается к базе данных;
4. получает или изменяет данные;
5. возвращает результат клиенту.

То есть **сервер выступает посредником** между клиентом и базой данных.

| Определение | Содержание |
|---|---|
| **База данных (Database)** | организованная совокупность данных, предназначенная для хранения и обработки информации |
| **СУБД (DBMS)** | программное обеспечение, обеспечивающее создание, хранение, изменение и получение данных из базы данных |

**Зачем это нужно в распределённом приложении:**

* данные переживают перезапуск сервера (персистентность);
* несколько клиентов работают с **одним** общим состоянием, а не с копиями в памяти;
* целостность и ограничения проверяет СУБД, а не только код приложения;
* групповые операции можно выполнить атомарно (транзакция);
* состояние можно наблюдать и восстанавливать независимо от процесса сервера.


---
## 2. База данных, СУБД и выбор СУБД

Сервер должен выполнять операции с БД **вместо** хранения основного состояния
предметной области только в памяти. Конкретный состав таблиц определяется
предметной областью. Промышленно сложная БД не требуется — нужна структура,
достаточная для выполнения операций ЛР.

**Допустимые СУБД** (любая, сочетающаяся с C#):

| СУБД | Плюсы для ЛР | Минусы для ЛР |
|---|---|---|
| **SQLite** | файл базы, нулевая установка, полная поддержка транзакций, `Microsoft.Data.Sqlite` | одна машина, ограниченная конкурентность на запись |
| **PostgreSQL** | промышленная, мощные транзакции и блокировки строк | требует установки сервера СУБД |
| **Microsoft SQL Server** | нативна для Windows/.NET, Management Studio | тяжёлая установка, объём ~ГБ |
| **MySQL** | популярна, понятный SQL | отдельная установка, особенности транзакций в старых версиях |

**Выбор для этой работы — SQLite**, обоснование:

1. база — обычный файл `flowershop.db`, никакой настройки сервера: эксперименты перезапускаются быстро и воспроизводимо;
2. `Microsoft.Data.Sqlite` — официальный ADO.NET-провайдер, работаёт с тем же C#-кодом, что и другие провайдеры;
3. поддерживаются `BEGIN` / `COMMIT` / `ROLLBACK`, то есть все эксперименты части 3 выполнимы;
4. сервер один и выполняет все запросы через своё соединение — типичное ограничение SQLite (одна запись) не мешает;
5. перенос на PostgreSQL/MS SQL сводится к замене строки подключения и пакета NuGet: SQL-инструкции используют стандартный синтаксис.

> Если в проекте требуется многопользовательская БД «по-настоящему», ставьте
> PostgreSQL — те же запросы и транзакции работают без изменения логики сервера.


---
## 3. Проектирование базы данных

Определяем сущности предметной области «Заказ и доставка цветов», которые
должны храниться в БД:

* **клиент** (`customers`) — кто оформляет заказ;
* **букет** (`bouquets`) — товар, у него есть **общий ресурс** `available_count`;
* **состав букета** (`bouquet_flowers`) — связь «многие ко многим» с цветами;
* **цветок** (`flowers`) — справочник цветов и цен;
* **заказ** (`orders`) — заказ клиента на букет, со статусом;
* **доставка** (`deliveries`) — адрес и курьер, связанные с заказом;
* **флорист** (`florists`) — исполнитель, назначаемый на заказ.

Для каждой таблицы задаём: **первичные ключи**, **атрибуты**, **внешние ключи**,
**типы данных**, **ограничения целостности** (`NOT NULL`, `UNIQUE`, `CHECK`,
`FOREIGN KEY`).


In [ ]:
-- Скрипт создания базы данных (SQLite): ЛР №4, Требование 3
PRAGMA foreign_keys = ON;

CREATE TABLE customers (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name   TEXT    NOT NULL,
    phone       TEXT    UNIQUE
);

CREATE TABLE flowers (
    id            INTEGER PRIMARY KEY AUTOINCREMENT,
    name          TEXT    NOT NULL UNIQUE,
    price_per_unit REAL    NOT NULL CHECK (price_per_unit >= 0)
);

CREATE TABLE bouquets (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    name            TEXT    NOT NULL,
    -- ОБЩИЙ РЕСУРС ЛР №2/№3/№4: доступные экземпляры букета
    available_count INTEGER NOT NULL DEFAULT 0 CHECK (available_count >= 0),
    florist_id      INTEGER REFERENCES florists(id) ON DELETE SET NULL
);

CREATE TABLE bouquet_flowers (
    bouquet_id INTEGER NOT NULL REFERENCES bouquets(id) ON DELETE CASCADE,
    flower_id  INTEGER NOT NULL REFERENCES flowers(id)  ON DELETE CASCADE,
    count      INTEGER NOT NULL CHECK (count > 0),
    PRIMARY KEY (bouquet_id, flower_id)
);

CREATE TABLE florists (
    id   INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL
);

CREATE TABLE orders (
    id             INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id    INTEGER NOT NULL REFERENCES customers(id),
    bouquet_id     INTEGER NOT NULL REFERENCES bouquets(id),
    delivery_date  TEXT    NOT NULL,
    status         TEXT    NOT NULL DEFAULT 'Created'
                   CHECK (status IN ('Created','BouquetPrepared',
                                     'FloristAssigned','DeliveryArranged','Cancelled')),
    delivery_address TEXT  NOT NULL
);

CREATE TABLE deliveries (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id   INTEGER NOT NULL UNIQUE REFERENCES orders(id) ON DELETE CASCADE,
    courier    TEXT    NOT NULL,
    address    TEXT    NOT NULL
);

CREATE INDEX idx_orders_customer ON orders(customer_id);


**Что здесь важно для защиты:**

* `CHECK (available_count >= 0)` — СУБД сама не даст ресурс уйти в минус, даже если
  в логике сервера допущена ошибка: это **второй** уровень защиты после `lock`;
* `FOREIGN KEY` — заказ не может ссылаться на несуществующий букет или клиента
  (ссылочная целостность);
* `UNIQUE` — одинаковый телефон клиента и одинаковое имя букета не задублируются;
* `PRIMARY KEY` — стабильный идентификатор записи, на который ссылается сервер.

Состав таблиц может отличаться в зависимости от варианта предметной области —
важно, чтобы её хватало для операций ЛР (чтение, изменение, общий ресурс,
составная операция).


---
## 4. Подключение сервера к БД

Подключение к БД выполняет **только сервер**:

```
Client
  ↓  (TCP, JSON — как в ЛР №1–№3)
Server   ← бизнес-логика, lock, транзакции
  ↓  (ADO.NET, SQL)
Database
```

Клиент не содержит строки подключения и не знает о существовании БД — иначе
нарушается и архитектура, и безопасность (клиенты начнут обходить бизнес-логику).

Минимальный набор операций с БД (Требование 5):

1. **чтение** — `GetBouquetStatus` / список заказов;
2. **добавление или изменение** — `CreateOrder` / `ReserveBouquet`;
3. **операция с общим ресурсом** — резервирование `available_count`.


In [ ]:
// Server/Data/Database.cs — подключение сервера к SQLite (ЛР №4, Требование 4)
using Microsoft.Data.Sqlite;

public sealed class Database : IDisposable
{
    private readonly string _connectionString =
        "Data Source=flowershop.db;Foreign Keys=True";

    /// <summary>Соединение принадлежит СЕРВЕРУ, а не клиенту.</summary>
    public SqliteConnection Open()
    {
        var connection = new SqliteConnection(_connectionString);
        connection.Open();
        // Включаем проверку внешних ключей для этой сессии
        using var cmd = connection.CreateCommand();
        cmd.CommandText = "PRAGMA foreign_keys = ON;";
        cmd.ExecuteNonQuery();
        return connection;
    }

    // ----- Чтение (операция 1) ------------------------------------
    public int GetAvailableCount(SqliteConnection cn, int bouquetId)
    {
        using var cmd = cn.CreateCommand();
        cmd.CommandText =
            "SELECT available_count FROM bouquets WHERE id = $id;";
        cmd.Parameters.AddWithValue("$id", bouquetId);
        var result = cmd.ExecuteScalar();
        if (result is null)
            throw new KeyNotFoundException($"Букет #{bouquetId} не найден");
        return Convert.ToInt32(result);
    }

    // ----- Изменение (операция 2) ---------------------------------
    public void InsertOrder(SqliteConnection cn, int customerId,
                            int bouquetId, string address)
    {
        using var cmd = cn.CreateCommand();
        cmd.CommandText =
            "INSERT INTO orders (customer_id, bouquet_id, delivery_date, " +
            "status, delivery_address) " +
            "VALUES ($c, $b, date('now'), 'Created', $a);";
        cmd.Parameters.AddWithValue("$c", customerId);
        cmd.Parameters.AddWithValue("$b", bouquetId);
        cmd.Parameters.AddWithValue("$a", address);
        cmd.ExecuteNonQuery();
    }

    public void Dispose() { /* приложение закрывает соединение при остановке */ }
}


---
## 5. Два уровня согласованности данных

В распределённом приложении несколько клиентов могут **одновременно** отправлять
серверу запросы, изменяющие одни и те же данные. Пусть в системе один доступный
ресурс `Available = 1`, и два клиента одновременно пытаются его получить.
Без синхронизации возможно некорректное поведение:

```
Client 1 → read Available = 1
Client 2 → read Available = 1      ← тот же старый ответ
Client 1 → reserve                 → Available = 0
Client 2 → reserve                 → Available = -1  (или ресурс выдан дважды)
```

В ЛР №3 такая проблема рассматривалась на уровне общего состояния приложения.
В ЛР №4 необходимо учитывать **два уровня**:

| Уровень | Что защищает | Механизм | Где выполняется |
|---|---|---|---|
| 1. Синхронизация выполнения кода сервера | последовательность «прочитать → проверить → изменить» в памяти процесса | **`lock`** (критическая секция) | код сервера, один процесс |
| 2. Согласованность изменений в БД | группа связанных SQL-операций как единое целое | **транзакция** (`COMMIT` / `ROLLBACK`) | СУБД, может быть несколько соединений/процессов |

Уровни не дублируют друг друга: `lock` не умеет отменять уже выполненные SQL-записи,
а транзакция не сериализует произвольный C#-код между запросами к БД.
Подробно о разнице — в разделе 8.


---
## 6. Критическая секция и механизм `lock`

**Критическая секция** — участок кода, в котором выполняются действия с общим
ресурсом, не допускающие одновременного выполнения несколькими потоками.

**`lock`** в C# гарантирует, что в каждый момент времени блокировку может держать
только один поток; остальные ждут выхода владельца. Компилятор разворачивает
`lock` в `Monitor.Enter` / `Monitor.Exit` (с `try/finally`), поэтому исключение
внутри блока не оставит блокировку навсегда.

Требуемая структура критической секции из методички:

```
lock (syncObject)
{
    check resource;
    if (resource available)
    {
        reserve resource;
        return SUCCESS;
    }
    return FAIL;
}
```

**Границы секции:** под `lock` помещаются *все взаимосвязанные действия,
которые должны выполняться атомарно* — проверка состояния ресурса и его
изменение. Весь метод под `lock` помещать **не нужно**: это превращает сервер
в последовательный обработчик и не даёт преимущества параллельной обработке.


In [ ]:
// Server/BusinessLogic.cs — критическая секция под lock (ЛР №3, действует и в ЛР №4)
private readonly object _syncObject = new();   // только для блокировки, не для данных

private Response ReserveWithLock(Bouquet bouquet, int bouquetId,
                                 RequestContext? context)
{
    lock (_syncObject)          // ← вход в критическую секцию
    {
        Log(context, "LOCK",  $"Букет #{bouquetId} — вход в критическую секцию");
        Log(context, "READ",  $"available_count = {bouquet.AvailableCount}");

        if (bouquet.AvailableCount <= 0)       // ПРОВЕРКА
        {
            Log(context, "FAIL", "Ресурс отсутствует");
            return Fail("Ресурс отсутствует"); // ← UNLOCK выполняется автоматически
        }

        bouquet.AvailableCount--;              // ИЗМЕНЕНИЕ — тут же, без гонки
        Log(context, "WRITE", $"available_count = {bouquet.AvailableCount}");
        Log(context, "SUCCESS", "Букет зарезервирован");
        return Ok(...);
    }                           // ← выход из критической секции (finally)
}


**Корректный результат при `Resource = 1` и двух клиентах:**

```
Client 1 → SUCCESS
Client 2 → FAIL
```

(или `FAIL` / `SUCCESS` — зависит от порядка захвата блокировки). Результат
`Client 1 → SUCCESS`, `Client 2 → SUCCESS` при одном ресурсе **некорректен**.

**Пример журнала выполнения** (Требование 5) — время, клиент, операция,
вход/выход из критической секции, результат, состояние ресурса:

```
[10:15:01.125] Client 1 → START Reserve
[10:15:01.130] Client 1 → LOCK
[10:15:01.131] Client 1 → Check resource
[10:15:01.135] Client 1 → SUCCESS
[10:15:01.140] Client 1 → UNLOCK
[10:15:01.141] Client 2 → START Reserve
[10:15:01.145] Client 2 → LOCK
[10:15:01.146] Client 2 → Check resource
[10:15:01.150] Client 2 → FAIL
[10:15:01.151] Client 2 → UNLOCK
```

Журнал по времени доказывает, что секции двух клиентов **не пересекаются**:
`LOCK` второго клиента зарегистрирован только после `UNLOCK` первого.


---
## 7. Транзакции, COMMIT и ROLLBACK

> **Транзакция (Transaction)** — последовательность операций с базой данных,
> рассматриваемая как единое целое, где все необходимые изменения выполняются
> успешно, либо изменения отменяются.
>
> **`COMMIT`** — подтверждение транзакции (изменения фиксируются).
> **`ROLLBACK`** — отмена транзакции (база возвращается к состоянию до `BEGIN`).

### 7.1 Что происходит без транзакции

Составная бизнес-операция «создать заказ + уменьшить остаток» состоит из двух
независимых SQL-операций. Если вторая падает, первая уже сохранена:

```
Create Order   → SUCCESS     ← запись в orders появилась
Update Stock   → ERROR       ← остаток не изменился
```

В базе остаётся заказ на букет, которого «нет в наличии» — **противоречивое
(частичное) состояние**. Каждая операция по отдельности корректна, но их
*сочетание* нарушает правило предметной области.

### 7.2 Что происходит с транзакцией

```
BEGIN
   ↓
Create Order      → SUCCESS
   ↓
Update Stock      → ERROR
   ↓
ROLLBACK          ← все изменения транзакции отменены
```

После `ROLLBACK` база возвращается к состоянию, существовавшему до начала
транзакции: нет ни заказа, ни изменённого остатка. Если бы ошибки не было,
операция завершилась бы `COMMIT`, и **обе** записи сохранились бы.

### 7.3 Свойства ACID

| Свойство | Значение | Проявление в ЛР |
|---|---|---|
| **Atomicity** (атомарность) | всё или ничего | ошибка между операциями → `ROLLBACK` |
| **Consistency** (согласованность) | база переходит только между корректными состояниями | `CHECK (available_count >= 0)` |
| **Isolation** (изоляция) | параллельные транзакции не мешают друг другу | два клиента не видят «полу-состояние» |
| **Durability** (долговечность) | зафиксированное сохраняется | после `COMMIT` данные не теряются |


In [ ]:
// Server/Data/OrderRepository.cs — составная операция в одной транзакции
// (Требования 2–5 части 3: COMMIT / ROLLBACK / моделирование ошибки)
using Microsoft.Data.Sqlite;

public sealed class OrderRepository
{
    private readonly Database _db;

    /// <summary>Флаг моделирует ошибку между операциями (Требование 2).</summary>
    public bool SimulateError { get; set; }

    /// <summary>Операция БЕЗ транзакции — для сравнения (эксперименты 1–2).</summary>
    public void PlaceOrderWithoutTransaction(int customerId, int bouquetId,
                                             string address)
    {
        using var cn = _db.Open();
        _db.InsertOrder(cn, customerId, bouquetId, address);   // 1. SUCCESS
        if (SimulateError)
            throw new InvalidOperationException("Искусственная ошибка"); // 2. ERROR
        DecrementStock(cn, bouquetId);                          // 2'. не выполнилось
        // Результат: заказ в базе ОСТАЛСЯ, остаток НЕ уменьшился → частичное состояние
    }

    /// <summary>Та же операция С транзакцией (эксперименты 3–4).</summary>
    public void PlaceOrderInTransaction(int customerId, int bouquetId,
                                        string address, SqliteTransaction tx)
    {
        using var cn = tx.Connection!;                 // BEGIN уже выполнен
        try
        {
            _db.InsertOrder(cn, customerId, bouquetId, address);  // Operation 1
            if (SimulateError)
                throw new InvalidOperationException("Искусственная ошибка");

            DecrementStock(cn, bouquetId, tx);                     // Operation 2

            tx.Commit();                       // COMMIT — обе операции сохранены
        }
        catch (Exception)
        {
            tx.Rollback();                     // ROLLBACK — база как до BEGIN
            throw;                             // ошибку пробрасываем наружу
        }
    }

    private void DecrementStock(SqliteConnection cn, int bouquetId,
                                SqliteTransaction tx)
    {
        using var cmd = cn.CreateCommand();
        cmd.Transaction = tx;                  // команда обязана принадлежать транзакции
        cmd.CommandText =
            "UPDATE bouquets SET available_count = available_count - 1 " +
            "WHERE id = $id AND available_count > 0;";
        cmd.Parameters.AddWithValue("$id", bouquetId);
        if (cmd.ExecuteNonQuery() == 0)
            throw new InvalidOperationException(
                "Ресурс отсутствует или запись не найдена");
    }
}


### 7.4 Четыре эксперимента части 3 (заполняются в таблице результатов)

| № | Транзакция | Операция 1 | Операция 2 | Ошибка | Состояние БД |
|---|---|---|---|---|---|
| 1 | нет | выполнена | выполнена | нет | **изменения сохранены** (и без транзакции обе операции корректны) |
| 2 | нет | выполнена | не выполнена | да | **частично изменено**: заказ есть, остаток не уменьшен |
| 3 | да | выполнена | выполнена | нет | **COMMIT** — изменения сохранены |
| 4 | да | выполнена | не выполнена | да | **ROLLBACK** — изменения отменены, состояние как до BEGIN |

Вопросы, которые нужно задать после эксперимента без транзакции:

1. Какая операция успела выполниться?
2. Какая операция не выполнилась?
3. Остались ли изменения в БД?
4. Возникло ли частичное изменение данных?
5. Является ли полученное состояние корректным?

**Ключевой вывод:** транзакция нужна не потому, что отдельные операции работают
неправильно, а потому, что *связанные* операции должны быть атомарны.


---
## 8. `lock` и транзакция: два разных механизма

| Критерий | `lock` | Транзакция |
|---|---|---|
| Что синхронизирует | выполнение **кода** в памяти процесса | группу **операций с БД** |
| Где выполняется | CLR, один процесс/поток | СУБД, любое число соединений и процессов |
| Что защищает | общую переменную/объект (`available_count` в памяти) | набор строк в таблицах |
| Отмена изменений | не умеет — изменение уже произошло | `ROLLBACK` возвращает состояние до `BEGIN` |
| Переживает падение процесса | нет | да (журнал транзакций СУБД) |
| Цена | потоки ждут на входе | блокировки строк СУБД, накладные расходы |
| Какая ошибка без него | гонка «прочитать → проверить → изменить» | частичное состояние при сбое второй операции |

> **`lock` не является заменой транзакции, а транзакция не является заменой `lock`.
> Они решают разные задачи и могут использоваться совместно.**

* **Может ли `lock` заменить транзакцию?** Нет. Блокировка защищает только пока
  жив процесс: она не откатит уже вставленный заказ, если сервер упадёт между
  операциями, и не защитит от другого процесса, пишущего в ту же базу.
* **Может ли транзакция заменить `lock`?** Не полностью. Транзакция объединяет
  SQL-операции, но *между* ними в C#-коде выполняется проверка и решение, и другой
  поток сервера может успеть прочитать устаревшее значение. Поэтому секция
  «прочитать → проверить → изменить → `COMMIT`» закрывается `lock`, а транзакция
  открывается **внутри** этой секции.
* В промышленных СУБД ту же задачу могут выполнять блокировки строк
  (`SELECT ... FOR UPDATE`) и уровни изоляции — это переносит защиту на уровень
  данных. В данной работе `lock` и транзакция разделены намеренно, чтобы
  наглядно показать **два разных уровня** согласованности.

**Правильный порядок в коде сервера:**
`lock` → чтение ресурса → проверка → составная операция в транзакции →
`COMMIT`/`ROLLBACK` → выход из `lock`.


---
## 9. Задание: четыре связанные части

### Часть 1. Подключение базы данных (1 ч)

* выбрать СУБД и **обосновать** выбор;
* создать базу данных, определить таблицы и связи;
* подключить сервер приложения к БД (клиент — нет);
* перенести основные данные предметной области в БД;
* реализовать операции чтения и изменения данных.

*Требования:* подготовка проекта (сохранить архитектуру ЛР №3), выбор СУБД,
проектирование БД (ПК, атрибуты, ВК, типы, ограничения), подключение сервера,
минимум три операции — чтение, добавление/изменение, операция с общим ресурсом.

### Часть 2. Конкурентный доступ к данным (2 ч)

* два клиента, общий ресурс из варианта (`Bouquet.AvailableCount`);
* конкурентное выполнение одной операции над одним ресурсом;
* определить критическую секцию и защитить её `lock`;
* проверить корректность результата и зафиксировать его.

### Часть 3. Реализация транзакции (2 ч)

* составная бизнес-операция: **создание заказа + уменьшение остатка**;
* минимум две связанные операции с БД;
* эксперимент **без** транзакции с искусственно вызванной ошибкой;
* исследовать состояние БД, ответить на 5 вопросов (раздел 7.4);
* повторить **с** транзакцией: `COMMIT` и `ROLLBACK`, сравнить результаты.

### Часть 4. Итоговое тестирование и анализ (1 ч)

* проверить двух клиентов, `lock`, транзакцию, `COMMIT`, `ROLLBACK`;
* повторный эксперимент и unit-тесты;
* заполнить таблицы результатов.

> **Unit-тесты (Требование 7)** должны подтверждать как минимум: корректность
> операции с общим ресурсом; невозможность получить один ресурс двумя
> конкурентными запросами; успешное выполнение составной операции; сохранение
> изменений после `COMMIT`; отсутствие частичных изменений после `ROLLBACK`.

> **Обработка ошибок (Требование 8):** сервер недоступен, соединение/подключение
> к БД отсутствует или некорректно, некорректный запрос, неизвестная операция,
> отсутствующий ресурс, попытка изменить несуществующую запись, попытка занять
> занятый ресурс, нарушение ограничения БД, ошибка внутри транзакции,
> невозможность `COMMIT`, необходимость `ROLLBACK`. Приложение не должно падать
> при ожидаемой ошибке.


---
## 10. Таблицы результатов (заполняются в отчёте)

### Таблица 1 — конкурентный доступ двух клиентов

| Механизм | Клиент | Время отправки запроса | Время входа в критическую секцию | Время выхода | Результат |
|---|---|---|---|---|---|
| Без синхронизации | Client 1 | | | | SUCCESS / FAIL |
| Без синхронизации | Client 2 | | | | |
| `lock` | Client 1 | | | | |
| `lock` | Client 2 | | | | |

*Результаты без синхронизации можно взять из ЛР №2.*

### Таблица 2 — эксперимент с транзакцией

| Эксперимент | Транзакция | Операция 1 | Операция 2 | Ошибка | Состояние БД |
|---|---|---|---|---|---|
| 1 | Нет | выполнена | выполнена | Нет | изменения сохранены |
| 2 | Нет | выполнена | не выполнена | Да | частично изменено |
| 3 | Да | выполнена | выполнена | Нет | COMMIT |
| 4 | Да | выполнена | не выполнена | Да | ROLLBACK |

### Таблица 3 — сравнение «без транзакции» и «с транзакцией»

| Критерий | Без транзакции | С транзакцией |
|---|---|---|
| Несколько связанных операций | | |
| Атомарность | | |
| Возможность частичного изменения | | |
| Использование COMMIT | | |
| Использование ROLLBACK | | |
| Сохранение изменений при успехе | | |
| Отмена изменений при ошибке | | |
| Согласованность результата | | |

### Таблица 4 — итоговый эксперимент

| Часть | Условие | Ожидаемый результат | Полученный результат |
|---|---|---|---|
| Конкурентный доступ | 2 клиента, Resource = 1 | Один SUCCESS, один FAIL | |
| `lock` | 2 клиента | Корректный результат | |
| Транзакция | Без ошибки | COMMIT | |
| Транзакция | С ошибкой | ROLLBACK | |
| БД | После COMMIT | Изменения сохранены | |
| БД | После ROLLBACK | Изменения отменены | |


---
## 11. Порядок проведения экспериментов (процедура для защиты)

### 11.1 Часть 1 — подключение БД

1. Создать базу `flowershop.db` скриптом из раздела 3.
2. Показать структуру: таблицы, ключи, ограничения (`schema` + `PRAGMA table_info`).
3. Запустить сервер — в журнале видно подключение к СУБД.
4. Клиент: `GetBouquetStatus` (чтение), `CreateOrder` (добавление),
   `ReserveBouquet` (общий ресурс) — данные берутся из БД, а не из памяти.

### 11.2 Часть 2 — конкурентный доступ

1. `AvailableCount = 1` (операция `SetBouquetAvailability`).
2. Запустить сервер (`--sync=lock`) и **двух клиентов**.
3. Оба клиента одновременно вызывают `ReserveBouquet`.
4. Показать в коде критическую секцию и объяснить, почему под `lock` именно она.
5. Сверить журнал: `LOCK → READ → WRITE/SUCCESS/FAIL → UNLOCK` без пересечений.
6. Заполнить таблицу 1 (для «без синхронизации» — режим `--sync=none` из ЛР №2/№3).

### 11.3 Часть 3 — транзакция

1. **Эксперимент без транзакции, без ошибки:** `CreateOrder` → `DecrementStock`,
   проверить БД (обе записи на месте).
2. **Без транзакции, с ошибкой:** включить `SimulateError`, выполнить — заказ
   остался, остаток не изменился → показать частичное состояние.
3. **С транзакцией, без ошибки:** `BEGIN → CreateOrder → DecrementStock → COMMIT`,
   проверить БД (обе записи).
4. **С транзакцией, с ошибкой:** `BEGIN → CreateOrder → ERROR → ROLLBACK`,
   проверить БД (изменений нет, состояние как до BEGIN).
5. Сравнить пункты 2 и 4 — заполнить таблицы 2 и 3.

### 11.4 Часть 4 — итоговое тестирование

```bash
dotnet build FlowerShop.sln
dotnet test  FlowerShop.sln          # unit-тесты, включая транзакции

# Терминал 1 — сервер
dotnet run --project Server -- 5000 --sync=lock

# Терминал 2 и 3 — два клиента
dotnet run --project Client -- 127.0.0.1 5000
dotnet run --project Client -- 127.0.0.1 5000
```

Проверить: общий ресурс → `lock` → транзакция → `COMMIT` → `ROLLBACK`, затем
заполнить таблицу 4.


---
## 12. Карта «требование → код → наблюдаемый результат»

| Требование | Реализация | Наблюдаемый результат |
|---|---|---|
| Тр. 2. Выбор СУБД | `Server/Data/Database.cs`, пакет `Microsoft.Data.Sqlite`, файл `flowershop.db` | обоснование + работающее подключение |
| Тр. 3. Проектирование БД | скрипт `schema.sql` (раздел 3) | таблицы с ПК/ВК/`CHECK`, `PRAGMA foreign_keys` |
| Тр. 4. Подключение сервера | `Database.Open()` в `Server` | клиент никогда не открывает соединение |
| Тр. 5. Операции с БД | `GetAvailableCount` (чтение), `InsertOrder`/`DecrementStock` (изменение), `ReserveBouquet` (ресурс) | данные читаются из БД |
| Ч.2 Тр. 3. `lock` | `BusinessLogic.ReserveWithLock`, объект `_syncObject` | журнал `LOCK/UNLOCK` без пересечений |
| Ч.2 Тр. 4. Проверка результата | `Resource = 1`, два клиента | один `SUCCESS`, один `FAIL` |
| Ч.3 Тр. 2. Без транзакции | `PlaceOrderWithoutTransaction` + `SimulateError` | частичное состояние в БД |
| Ч.3 Тр. 3–5. Транзакция | `PlaceOrderInTransaction`: `Commit()` / `Rollback()` | `COMMIT` — изменения есть, `ROLLBACK` — нет |
| Тр. 7. Unit-тесты | `Server.Tests/TransactionTests.cs`, `SynchronizationTests.cs` | зелёный `dotnet test` |
| Тр. 8. Обработка ошибок | `try/catch` + `ErrorCodes` в `Shared/Protocol` | клиент получает текст ошибки, сервер не падает |


---
## 13. Ответы на контрольные вопросы (1–8)

1. **Что такое база данных?** — Организованная совокупность данных,
   предназначенная для хранения и обработки информации.
2. **Что такое СУБД?** — Программное обеспечение, обеспечивающее создание,
   хранение, изменение и получение данных из базы данных (SQLite, PostgreSQL,
   MySQL, MS SQL Server).
3. **Для чего в распределённом приложении используется база данных?** — Чтобы
   состояние предметной области хранилось персистентно и в одном месте, а не в
   памяти сервера: переживает перезапуск, доступна всем клиентам через сервер,
   целостность контролирует СУБД, групповые изменения выполняются атомарно.
4. **Что такое конкурентный доступ к данным?** — Одновременные или пересекающиеся
   по времени обращения нескольких клиентов/потоков к одним и тем же данным.
5. **Какая проблема может возникнуть, если два клиента одновременно изменяют один
   ресурс?** — Race condition: оба читают `Available = 1` до изменения и оба
   резервируют — ресурс выдан дважды либо счётчик ушёл в минус. Нарушается
   согласованность данных.
6. **Для чего используется `lock`?** — Для взаимного исключения: чтобы код
   «прочитать → проверить → изменить» выполнялся в критической секции только одним
   потоком, остальные ждут выхода.
7. **Что такое критическая секция?** — Участок программы, в котором выполняются
   действия с общим ресурсом, не допускающие одновременного выполнения несколькими
   потоками.
8. **Почему проверка состояния ресурса и его изменение должны выполняться
   согласованно?** — Между ними возникает окно гонки: другой поток успевает
   прочитать устаревшее значение и принять то же решение. Проверка и изменение
   должны быть одной неделимой последовательностью (одной критической секцией).


---
## 13.1 Ответы на контрольные вопросы (9–16)

9. **Почему нельзя без необходимости помещать под `lock` весь метод?** — Это
   узкое место: все клиенты сериализуются, падает пропускная способность, растёт
   время ожидания и риск взаимных блокировок. Под `lock` идут только действия с
   общим ресурсом, которые должны быть атомарны.
10. **Что такое транзакция?** — Последовательность операций с БД, рассматриваемая
    как единое целое: либо все выполняются успешно, либо изменения отменяются.
11. **Для чего используется `COMMIT`?** — Подтверждает транзакцию: все её изменения
    фиксируются в базе и становятся видимыми другим соединениям.
12. **Для чего используется `ROLLBACK`?** — Отменяет транзакцию: база возвращается
    к состоянию, существовавшему до `BEGIN`.
13. **Что произойдёт, если одна из операций составной бизнес-операции завершится
    ошибкой без использования транзакции?** — Возникнет частичное (противоречивое)
    состояние: например, заказ создан, а остаток не уменьшен. Каждая операция
    корректна по отдельности, но вместе они нарушают правило предметной области.
14. **В чём разница между `lock` и транзакцией?** — `lock` синхронизирует выполнение
    кода в пределах одного процесса (защита от гонки в памяти); транзакция
    обеспечивает атомарность группы операций с БД на уровне СУБД (защита от
    частичных изменений, переживает сбой, умеет `ROLLBACK`).
15. **Может ли `lock` заменить транзакцию базы данных? Почему?** — Нет. Блокировка
    действует только в живом процессе: она не откатит частично записанные данные при
    падении сервера и не защитит от другого процесса, работающего с той же базой.
16. **Может ли транзакция заменить `lock` в критической секции? Почему?** — Не
    полностью. Транзакция объединяет SQL-операции, но не сериализует произвольный
    C#-код между запросами: другой поток может прочитать устаревшее значение между
    чтением и решением. Поэтому «прочитать → проверить → изменить → `COMMIT`»
    закрывается `lock`, а транзакция выполняется внутри секции.


---
## 14. Ключевые выводы

1. **Сервер — посредник между клиентом и БД.** Клиент не знает о базе данных;
   все чтения и изменения идут через бизнес-логику сервера.
2. **Два уровня — две проблемы.** `lock` защищает выполнение кода (гонка в памяти),
   транзакция — согласованность данных (частичные изменения в БД). Один механизм
   не заменяет другой.
3. **Критическая секция минимальна и точно определена:** проверка ресурса и его
   изменение. Весь метод под `lock` не помещается.
4. **Корректный конкурентный результат при `Resource = 1`** — ровно один `SUCCESS`
   и один `FAIL`; `SUCCESS`+`SUCCESS` означает ошибку синхронизации.
5. **Транзакция даёт «всё или ничего»:** без неё составная операция оставляет
   частичное состояние, с `COMMIT` изменения сохраняются, с `ROLLBACK` — полностью
   отменяются.
6. **Структура БД — вторая линия защиты:** `CHECK`, `FOREIGN KEY`, `UNIQUE` не
   допускают некорректных данных даже при ошибке в коде.
7. **Практический итог:** распределённое приложение с БД требует синхронизации на
   уровне потоков сервера **и** атомарности на уровне СУБД — обе защиты видны в
   журнале, таблицах результатов и unit-тестах.
